
## Objective
This notebook focuses on graph-based candidate model training in the recommendation system pipeline.

## Data Sources
Primary inputs referenced in this notebook include:
- main_data.csv
- item_catalog.csv
- user_item_interactions.csv
- user_id_mapping.json
- item_id_mapping.json
- reverse_item_id_mapping.json
- ngcf_top_candidates.csv
- ngcf_training_summary.json

## Core Workflow
- Import dependencies and initialize configuration.
- Load source datasets into dataframes.
- Join data from multiple sources to build training context.
- Aggregate events/items to create behavioral features.
- Build or use embedding representations for items/categories.
- Train model components using prepared features.
- Persist model/artifacts for reuse.

## Expected Outputs
- Processed CSV outputs for downstream stages.
- Serialized model/artifact files for runtime loading.
- Trained model objects and related metrics/logs.
- Prediction/score outputs for ranking or evaluation.

## Role in Pipeline
This notebook contributes notebook-specific assets/signals that feed later candidate generation, ranking, or retraining steps.



<!-- AUTO_CELL_EXPLANATION -->
### Cell 1: # Run this only if torch is not installed
**Explanation:** This cell executes logic related to `# Run this only if torch is not installed`.

**Possible Output:** Possible output: text/log/value print.


In [1]:
# Run this only if torch is not installed
!pip install torch

Defaulting to user installation because normal site-packages is not writeable


<!-- AUTO_CELL_EXPLANATION -->
### Cell 2: import json
**Explanation:**

- Imports libraries for:
  - Data processing (pandas, numpy)
  - Deep learning (**PyTorch**)
  - Serialization (json, pickle)
- Sets **random seeds** for reproducibility.
- Defines file paths for:
  - Input data (`main_data.csv`, item catalog)
  - NGCF outputs (model, embeddings, mappings)
- Prepares storage for:
  - User-item interactions
  - ID mappings
  - Trained model & embeddings
- Detects computation device (**CPU/GPU**).

**Possible Output:** Possible output:

- Device: cpu
- Main data exists: True
- Item catalog exists: True
- Output folder: C:\D drive\item_recommendation_model_create\data\ngcf_output


In [2]:
import json
import pickle
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

BASE_DIR = Path(r"C:\D drive\item_recommendation_model_create")
DATA_DIR = BASE_DIR / "data"

MAIN_DATA_FILE = DATA_DIR / "main_data.csv"
ITEM_CATALOG_FILE = DATA_DIR / "item_catalog_output" / "item_catalog.csv"

NGCF_OUTPUT_DIR = DATA_DIR / "ngcf_output"
NGCF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INTERACTION_FILE = NGCF_OUTPUT_DIR / "user_item_interactions.csv"
USER_MAPPING_FILE = NGCF_OUTPUT_DIR / "user_id_mapping.json"
ITEM_MAPPING_FILE = NGCF_OUTPUT_DIR / "item_id_mapping.json"
REVERSE_ITEM_MAPPING_FILE = NGCF_OUTPUT_DIR / "reverse_item_id_mapping.json"

MODEL_FILE = NGCF_OUTPUT_DIR / "ngcf_model.pt"
USER_EMBEDDING_FILE = NGCF_OUTPUT_DIR / "ngcf_user_embeddings.npy"
ITEM_EMBEDDING_FILE = NGCF_OUTPUT_DIR / "ngcf_item_embeddings.npy"
TOP_CANDIDATE_FILE = NGCF_OUTPUT_DIR / "ngcf_top_candidates.csv"
TRAINING_SUMMARY_FILE = NGCF_OUTPUT_DIR / "ngcf_training_summary.json"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Main data exists:", MAIN_DATA_FILE.exists())
print("Item catalog exists:", ITEM_CATALOG_FILE.exists())
print("Output folder:", NGCF_OUTPUT_DIR)

Device: cpu
Main data exists: True
Item catalog exists: True
Output folder: C:\D drive\item_recommendation_model_create\data\ngcf_output


<!-- AUTO_CELL_EXPLANATION -->
### Cell 3: df = pd.read_csv(MAIN_DATA_FILE)
**Explanation:**

- Loads:
  - **Main transaction data** (`main_data.csv`)
  - **Clean item catalog** (`item_catalog.csv`)
- Cleans column names (removes extra spaces).
- Prints dataset shapes for validation.
- Displays sample rows for quick inspection.


**Possible Output:** Possible output:
Main data shape: (10000, 10)
Item catalog shape: (1200, 11)


In [3]:
df = pd.read_csv(MAIN_DATA_FILE)
item_catalog_df = pd.read_csv(ITEM_CATALOG_FILE)

df.columns = [c.strip() for c in df.columns]
item_catalog_df.columns = [c.strip() for c in item_catalog_df.columns]

print("Main data shape:", df.shape)
print("Item catalog shape:", item_catalog_df.shape)

display(df.head())
display(item_catalog_df.head())

Main data shape: (1000000, 16)
Item catalog shape: (229, 11)


,transactionId,customerId,customerName,customerPersona,itemId,itemName,category,quantity,orderDate,isHoliday,isFestival,season,timeSlot,dayOfWeek,hour,month
0,1,23433,MD MOSSIN,family_grocery,952,Chashi Aroma.Chinigura Rice-2kg,Pantry-Rice,6,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
1,1,23433,MD MOSSIN,family_grocery,453,Saad Testy Bit Salt-100gm,pantry salt,1,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
2,1,23433,MD MOSSIN,family_grocery,15262,Cow Brain-K,Meat-Fresh,2,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
3,1,23433,MD MOSSIN,family_grocery,32441,Ela vista pomace olive oil 250 ml,Pantry-Oils,1,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1
4,1,23433,MD MOSSIN,family_grocery,13986,Cow Stomach-K,Meat-Fresh,2,2025-01-01 15:40:00,0,1,Winter,Afternoon,Wednesday,15,1


,itemId,itemName,category,avgPrice,minPrice,maxPrice,avgQuantity,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag
0,13,Lemon Bright Dish Wash 250ml B2 G1,Household-Kitchen,NaN,NaN,NaN,1.40,3052,1,1,0
1,27,Pran Toast-250g,Snacks-General,NaN,NaN,NaN,2.16,2122,1,1,0
2,109,Wheel 2in1 Clean & Fresh Powder-1Kg,Household-Laundry,NaN,NaN,NaN,1.39,2085,1,1,0
3,125,Wheel Laundry Soap 125g,Personal-Care-Bath,NaN,NaN,NaN,1.12,2820,1,1,0
4,129,Vim Bar Lemon-125gm,Household-Kitchen,NaN,NaN,NaN,1.37,2816,1,1,0


<!-- AUTO_CELL_EXPLANATION -->
### Cell 4: required_cols = ["customerId", "itemId", "quantity"]
**Explanation:**
- Validates required columns:
  - Main data → `customerId`, `itemId`, `quantity`
  - Catalog → `itemId`, `itemName`, `category`
- Raises error if any column is missing.
- Cleans and standardizes:
  - Converts IDs to **integer**
  - Converts `quantity` to numeric (default = 1 if missing)
  - Strips text fields in catalog
- Ensures both datasets are **ready for model training**.

**Possible Output:** Possible output: Validation completed


In [4]:
required_cols = ["customerId", "itemId", "quantity"]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns in main_data.csv: {missing_cols}")

catalog_required_cols = ["itemId", "itemName", "category"]

missing_catalog_cols = [col for col in catalog_required_cols if col not in item_catalog_df.columns]

if missing_catalog_cols:
    raise ValueError(f"Missing required columns in item_catalog.csv: {missing_catalog_cols}")

df = df.copy()
item_catalog_df = item_catalog_df.copy()

df["customerId"] = df["customerId"].astype(int)
df["itemId"] = df["itemId"].astype(int)
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce").fillna(1)

item_catalog_df["itemId"] = item_catalog_df["itemId"].astype(int)
item_catalog_df["itemName"] = item_catalog_df["itemName"].astype(str).str.strip()
item_catalog_df["category"] = item_catalog_df["category"].astype(str).str.strip()

print("Validation completed")

Validation completed


<!-- AUTO_CELL_EXPLANATION -->
### Cell 5: interaction_df = (
**Explanation:**

- Aggregates transactions into **user-item interactions**:
  - `purchaseCount` → number of times item was bought
  - `totalQuantity` → total quantity purchased
- Creates binary interaction signal (`interaction = 1`)
- Joins with item catalog to add:
  - `itemName`, `category`
- Saves final interaction dataset for **NGCF training**


**Possible Output:** Possible output:

- Interaction shape: (8000, 6)
- Unique users: 500
- Unique items: 1200


In [5]:
interaction_df = (
    df.groupby(["customerId", "itemId"], as_index=False)
      .agg(
          purchaseCount=("itemId", "count"),
          totalQuantity=("quantity", "sum")
      )
)

interaction_df["interaction"] = 1

interaction_df = interaction_df.merge(
    item_catalog_df[["itemId", "itemName", "category"]],
    on="itemId",
    how="left"
)

interaction_df.to_csv(INTERACTION_FILE, index=False)

print("Interaction shape:", interaction_df.shape)
print("Unique users:", interaction_df["customerId"].nunique())
print("Unique items:", interaction_df["itemId"].nunique())

display(interaction_df.head())

Interaction shape: (42135, 7)
Unique users: 185
Unique items: 229


,customerId,itemId,purchaseCount,totalQuantity,interaction,itemName,category
0,2242,13,8,12,1,Lemon Bright Dish Wash 250ml B2 G1,Household-Kitchen
1,2242,27,4,6,1,Pran Toast-250g,Snacks-General
2,2242,109,9,13,1,Wheel 2in1 Clean & Fresh Powder-1Kg,Household-Laundry
3,2242,125,15,18,1,Wheel Laundry Soap 125g,Personal-Care-Bath
4,2242,129,7,7,1,Vim Bar Lemon-125gm,Household-Kitchen


<!-- AUTO_CELL_EXPLANATION -->
### Cell 6: unique_users = sorted(interaction_df["customerId"].unique().tolist())
**Explanation:** - Extracts unique:
  - **Users** from interaction data
  - **Items** from item catalog
- Creates mappings:
  - `userId → index`
  - `itemId → index`
  - Reverse mapping for items
- Saves mappings as **JSON files**
- Prepares data for **matrix-based model training**

**Possible Output:** Possible output:
- Number of users: 500
- Number of items: 1200


In [6]:
unique_users = sorted(interaction_df["customerId"].unique().tolist())
unique_items = sorted(item_catalog_df["itemId"].unique().tolist())

user_id_to_index = {int(user_id): idx for idx, user_id in enumerate(unique_users)}
user_index_to_id = {idx: int(user_id) for user_id, idx in user_id_to_index.items()}

item_id_to_index = {int(item_id): idx for idx, item_id in enumerate(unique_items)}
item_index_to_id = {idx: int(item_id) for item_id, idx in item_id_to_index.items()}

with open(USER_MAPPING_FILE, "w", encoding="utf-8") as f:
    json.dump(user_id_to_index, f, ensure_ascii=False, indent=4)

with open(ITEM_MAPPING_FILE, "w", encoding="utf-8") as f:
    json.dump(item_id_to_index, f, ensure_ascii=False, indent=4)

with open(REVERSE_ITEM_MAPPING_FILE, "w", encoding="utf-8") as f:
    json.dump(item_index_to_id, f, ensure_ascii=False, indent=4)

num_users = len(user_id_to_index)
num_items = len(item_id_to_index)

print("Number of users:", num_users)
print("Number of items:", num_items)

Number of users: 185
Number of items: 229


<!-- AUTO_CELL_EXPLANATION -->
### Cell 7: indexed_rows = []
**Explanation:**
- Converts raw IDs → **indexed format** using mapping dictionaries:
  - `customerId → user_idx`
  - `itemId → item_idx`
- Filters out unmatched users/items.
- Creates `indexed_interaction_df` with:
  - Indexed IDs + interaction stats
- Builds:
  - `user_positive_items` → items each user interacted with
  - `all_item_indices` → full item space
- Prepares data for **negative sampling & training**
**Possible Output:** Possible output:
- Indexed interaction shape: (8000, 6)


In [7]:
indexed_rows = []

for _, row in interaction_df.iterrows():
    customer_id = int(row["customerId"])
    item_id = int(row["itemId"])
    
    if customer_id not in user_id_to_index:
        continue
    
    if item_id not in item_id_to_index:
        continue
    
    indexed_rows.append({
        "user_idx": user_id_to_index[customer_id],
        "item_idx": item_id_to_index[item_id],
        "customerId": customer_id,
        "itemId": item_id,
        "purchaseCount": int(row["purchaseCount"]),
        "totalQuantity": float(row["totalQuantity"])
    })

indexed_interaction_df = pd.DataFrame(indexed_rows)

user_positive_items = defaultdict(set)

for _, row in indexed_interaction_df.iterrows():
    user_positive_items[int(row["user_idx"])].add(int(row["item_idx"]))

all_item_indices = set(range(num_items))

print("Indexed interaction shape:", indexed_interaction_df.shape)
display(indexed_interaction_df.head())

Indexed interaction shape: (42135, 6)


,user_idx,item_idx,customerId,itemId,purchaseCount,totalQuantity
0,0,0,2242,13,8,12.0
1,0,1,2242,27,4,6.0
2,0,2,2242,109,9,13.0
3,0,3,2242,125,15,18.0
4,0,4,2242,129,7,7.0


<!-- AUTO_CELL_EXPLANATION -->
### Cell 8: def build_normalized_adj(num_users, num_items, interactions):
**Explanation:**
 Graph Construction (Normalized Adjacency Matrix - NGCF)



- Builds a **bipartite graph**:
  - Users ↔ Items
- Creates adjacency matrix where:
  - Users and items are treated as nodes
- Converts to **sparse tensor (PyTorch)** for efficiency
- Applies **symmetric normalization**:
  - Uses degree-based scaling
- Outputs `norm_adj` → core graph structure for NGCF

**Possible Output:** Possible output:
- Normalized adjacency created
- Shape: torch.Size([1700, 1700])
- Edges: 16000


In [8]:
def build_normalized_adj(num_users, num_items, interactions):
    rows = []
    cols = []

    for user_idx, item_idx in interactions:
        user_node = int(user_idx)
        item_node = int(num_users + item_idx)

        rows.append(user_node)
        cols.append(item_node)

        rows.append(item_node)
        cols.append(user_node)

    node_count = num_users + num_items

    indices = torch.LongTensor([rows, cols])
    values = torch.ones(len(rows), dtype=torch.float32)

    adj = torch.sparse_coo_tensor(
        indices,
        values,
        size=(node_count, node_count)
    ).coalesce()

    degree = torch.sparse.sum(adj, dim=1).to_dense()
    degree_inv_sqrt = torch.pow(degree + 1e-8, -0.5)

    row_idx = adj.indices()[0]
    col_idx = adj.indices()[1]

    norm_values = degree_inv_sqrt[row_idx] * adj.values() * degree_inv_sqrt[col_idx]

    norm_adj = torch.sparse_coo_tensor(
        adj.indices(),
        norm_values,
        size=adj.shape
    ).coalesce()

    return norm_adj


interaction_pairs = indexed_interaction_df[["user_idx", "item_idx"]].values.tolist()

norm_adj = build_normalized_adj(
    num_users=num_users,
    num_items=num_items,
    interactions=interaction_pairs
).to(DEVICE)

print("Normalized adjacency created")
print("Shape:", norm_adj.shape)
print("Edges:", norm_adj._nnz())

Normalized adjacency created
Shape: torch.Size([414, 414])
Edges: 84270


C:\Users\HP\AppData\Local\Temp\ipykernel_19440\3136149545.py:20: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  adj = torch.sparse_coo_tensor(


<!-- AUTO_CELL_EXPLANATION -->
### Cell 9: class NGCFStyleLightGCN(nn.Module):
**Explanation:**

- Defines **graph-based recommendation model** (`NGCFStyleLightGCN`):
  - Learns **user & item embeddings**
- Key components:
  - Embedding layers for users and items
  - **Graph propagation** using normalized adjacency matrix
  - Multi-layer message passing
  - Final embedding = **average of all layer outputs**
- `forward()`:
  - Computes **positive vs negative scores**
- Uses **BPR loss**:
  - Optimizes ranking (positive > negative)

**Possible Output:** Possible output: text/log/value print.

In [9]:
class NGCFStyleLightGCN(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64, num_layers=3):
        super().__init__()
        
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers
        
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight)
    
    def propagate(self, norm_adj):
        user_emb = self.user_embedding.weight
        item_emb = self.item_embedding.weight
        
        all_emb = torch.cat([user_emb, item_emb], dim=0)
        embeddings = [all_emb]
        
        for _ in range(self.num_layers):
            all_emb = torch.sparse.mm(norm_adj, all_emb)
            embeddings.append(all_emb)
        
        final_emb = torch.mean(torch.stack(embeddings, dim=0), dim=0)
        
        final_user_emb = final_emb[:self.num_users]
        final_item_emb = final_emb[self.num_users:]
        
        return final_user_emb, final_item_emb
    
    def forward(self, user_idx, positive_item_idx, negative_item_idx, norm_adj):
        final_user_emb, final_item_emb = self.propagate(norm_adj)
        
        user_vec = final_user_emb[user_idx]
        pos_vec = final_item_emb[positive_item_idx]
        neg_vec = final_item_emb[negative_item_idx]
        
        pos_score = torch.sum(user_vec * pos_vec, dim=1)
        neg_score = torch.sum(user_vec * neg_vec, dim=1)
        
        return pos_score, neg_score
    
    def get_embeddings(self, norm_adj):
        return self.propagate(norm_adj)


def bpr_loss(pos_score, neg_score):
    return torch.mean(-torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8))


print("Model class ready")

Model class ready


<!-- AUTO_CELL_EXPLANATION -->
### Cell 10: def sample_training_batch(batch_size):
**Explanation:**

- Generates training batches for **BPR loss**:
  - Selects a random **user**
  - Picks a **positive item** (already interacted)
  - Samples a **negative item** (not interacted)
- Ensures:
  - Negative item is not in user's history
  - Retries up to 50 times if conflict occurs
- Returns tensors:
  - `users`, `positive_items`, `negative_items`
- Moves data to **CPU/GPU (DEVICE)**


**Possible Output:** Possible output:
Negative sampler ready

In [10]:
def sample_training_batch(batch_size):
    users = []
    positive_items = []
    negative_items = []
    
    valid_users = list(user_positive_items.keys())
    
    while len(users) < batch_size:
        user_idx = random.choice(valid_users)
        
        pos_items = list(user_positive_items[user_idx])
        
        if not pos_items:
            continue
        
        positive_item_idx = random.choice(pos_items)
        
        negative_item_idx = random.randint(0, num_items - 1)
        
        retry_count = 0
        
        while negative_item_idx in user_positive_items[user_idx] and retry_count < 50:
            negative_item_idx = random.randint(0, num_items - 1)
            retry_count += 1
        
        if negative_item_idx in user_positive_items[user_idx]:
            continue
        
        users.append(user_idx)
        positive_items.append(positive_item_idx)
        negative_items.append(negative_item_idx)
    
    return (
        torch.LongTensor(users).to(DEVICE),
        torch.LongTensor(positive_items).to(DEVICE),
        torch.LongTensor(negative_items).to(DEVICE)
    )


print("Negative sampler ready")

Negative sampler ready


<!-- AUTO_CELL_EXPLANATION -->
### Cell 11: EMBEDDING_DIM = 64
**Explanation:**

- Sets training hyperparameters:
  - Embedding size, layers, epochs, batch size
- Initializes:
  - NGCF model
  - Adam optimizer
- Training loop:
  - Samples batches (user, positive, negative)
  - Computes scores via model
  - Applies **BPR loss**
  - Backpropagation + weight update
- Tracks **loss per epoch**
- Prints progress periodically

**Possible Output:** Possible output:

- Epoch 1/80, Loss: 0.693245
- Epoch 10/80, Loss: 0.512378
- Epoch 20/80, Loss: 0.421905
- Epoch 30/80, Loss: 0.356210
...
- Epoch 80/80, Loss: 0.210543

NGCF training completed


In [11]:
EMBEDDING_DIM = 64
NUM_LAYERS = 3
EPOCHS = 80
BATCH_SIZE = 2048
STEPS_PER_EPOCH = max(20, len(indexed_interaction_df) // BATCH_SIZE)
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5

model = NGCFStyleLightGCN(
    num_users=num_users,
    num_items=num_items,
    embedding_dim=EMBEDDING_DIM,
    num_layers=NUM_LAYERS
).to(DEVICE)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

loss_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_losses = []
    
    for step in range(STEPS_PER_EPOCH):
        user_batch, pos_batch, neg_batch = sample_training_batch(BATCH_SIZE)
        
        optimizer.zero_grad()
        
        pos_score, neg_score = model(
            user_batch,
            pos_batch,
            neg_batch,
            norm_adj
        )
        
        loss = bpr_loss(pos_score, neg_score)
        loss.backward()
        optimizer.step()
        
        epoch_losses.append(float(loss.item()))
    
    avg_loss = float(np.mean(epoch_losses))
    loss_history.append(avg_loss)
    
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch}/{EPOCHS}, Loss: {avg_loss:.6f}")

print("NGCF training completed")

Epoch 1/80, Loss: 0.690744
Epoch 10/80, Loss: 0.344730
Epoch 20/80, Loss: 0.196560
Epoch 30/80, Loss: 0.163601
Epoch 40/80, Loss: 0.149495
Epoch 50/80, Loss: 0.130050
Epoch 60/80, Loss: 0.114446
Epoch 70/80, Loss: 0.094881
Epoch 80/80, Loss: 0.078019
NGCF training completed


<!-- AUTO_CELL_EXPLANATION -->
### Cell 12: model.eval()
**Explanation:**

- Switches model to **evaluation mode**
- Generates:
  - Final **user embeddings**
  - Final **item embeddings**
- Converts embeddings → NumPy arrays
- Saves:
  - Embeddings (`.npy` files)
  - Full model (`.pt`) including:
    - Model weights
    - Metadata (users, items, mappings, config)
- Prepares artifacts for **inference & recommendation**

**Possible Output:** Possible output:

- Saved model: C:\D drive\item_recommendation_model_create\data\ngcf_output\ngcf_model.pt
- Saved user embeddings: C:\D drive\item_recommendation_model_create\data\ngcf_output\ngcf_user_embeddings.npy
- Saved item embeddings: C:\D drive\item_recommendation_model_create\data\ngcf_output\ngcf_item_embeddings.npy


In [12]:
model.eval()

with torch.no_grad():
    user_embeddings, item_embeddings = model.get_embeddings(norm_adj)
    user_embeddings_np = user_embeddings.detach().cpu().numpy()
    item_embeddings_np = item_embeddings.detach().cpu().numpy()

np.save(USER_EMBEDDING_FILE, user_embeddings_np)
np.save(ITEM_EMBEDDING_FILE, item_embeddings_np)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "num_users": num_users,
        "num_items": num_items,
        "embedding_dim": EMBEDDING_DIM,
        "num_layers": NUM_LAYERS,
        "user_id_to_index": user_id_to_index,
        "item_id_to_index": item_id_to_index,
        "item_index_to_id": item_index_to_id
    },
    MODEL_FILE
)

print("Saved model:", MODEL_FILE)
print("Saved user embeddings:", USER_EMBEDDING_FILE)
print("Saved item embeddings:", ITEM_EMBEDDING_FILE)

Saved model: C:\D drive\item_recommendation_model_create\data\ngcf_output\ngcf_model.pt
Saved user embeddings: C:\D drive\item_recommendation_model_create\data\ngcf_output\ngcf_user_embeddings.npy
Saved item embeddings: C:\D drive\item_recommendation_model_create\data\ngcf_output\ngcf_item_embeddings.npy


<!-- AUTO_CELL_EXPLANATION -->
### Cell 13: TOP_N_PER_USER = 200
**Explanation:**

- Normalizes **user & item embeddings** (cosine similarity ready)
- Computes **user–item similarity scores** using dot product
- For each user:
  - Selects **Top-N items** (`TOP_N_PER_USER`)
  - Assigns rank based on score
- Processes users in batches for efficiency
- Merges item metadata (name, category)
- Saves final **candidate recommendation list**

**Possible Output:** Possible output:

- NGCF candidate shape: (100000, 5)
- Saved: C:\D drive\item_recommendation_model_create\data\ngcf_output\ngcf_top_candidates.csv


In [13]:
TOP_N_PER_USER = 200
BATCH_USER_SIZE = 128

item_norm = item_embeddings_np / (np.linalg.norm(item_embeddings_np, axis=1, keepdims=True) + 1e-8)
user_norm = user_embeddings_np / (np.linalg.norm(user_embeddings_np, axis=1, keepdims=True) + 1e-8)

candidate_rows = []

for start in range(0, num_users, BATCH_USER_SIZE):
    end = min(start + BATCH_USER_SIZE, num_users)
    
    batch_user_emb = user_norm[start:end]
    score_matrix = batch_user_emb.dot(item_norm.T)
    
    for local_idx, user_idx in enumerate(range(start, end)):
        customer_id = user_index_to_id[user_idx]
        scores = score_matrix[local_idx]
        
        top_indices = np.argsort(scores)[::-1][:TOP_N_PER_USER]
        
        for rank, item_idx in enumerate(top_indices, start=1):
            item_id = item_index_to_id[int(item_idx)]
            
            candidate_rows.append({
                "customerId": int(customer_id),
                "itemId": int(item_id),
                "ngcf_score": float(scores[item_idx]),
                "ngcf_rank": int(rank)
            })

ngcf_top_candidates_df = pd.DataFrame(candidate_rows)

ngcf_top_candidates_df = ngcf_top_candidates_df.merge(
    item_catalog_df[["itemId", "itemName", "category"]],
    on="itemId",
    how="left"
)

ngcf_top_candidates_df.to_csv(TOP_CANDIDATE_FILE, index=False)

print("NGCF candidate shape:", ngcf_top_candidates_df.shape)
print("Saved:", TOP_CANDIDATE_FILE)

display(ngcf_top_candidates_df.head(20))

NGCF candidate shape: (37000, 6)
Saved: C:\D drive\item_recommendation_model_create\data\ngcf_output\ngcf_top_candidates.csv


,customerId,itemId,ngcf_score,ngcf_rank,itemName,category
0,2242,26701,0.999397,1,Nescafe Classic Jar-25gm,Beverage-Hot
1,2242,3554,0.999288,2,Parachute Skin Pure Aloe Vera Gel- 50ml,Personal-Care-Cosmetics
2,2242,7075,0.999280,3,Herman Peanut Butter-340gm,Dairy-Other
3,2242,30788,0.999278,4,Teer Atta 2kg,Pantry-Flour
4,2242,3182,0.999265,5,Starship Chocolate Milk-200ml,Dairy-Milk
5,2242,13922,0.999258,6,Rui Fish (S1)-(MNK),Fish-Fresh
6,2242,4122,0.999236,7,Clear Complete Active Care Shampoo-330ml,Personal-Care-Hair
7,2242,976,0.999234,8,Lata Rice-KG,Pantry-Rice
8,2242,3185,0.999231,9,Marks Milk Shake AS-125,Dairy-Milk
9,2242,125,0.999219,10,Wheel Laundry Soap 125g,Personal-Care-Bath


<!-- AUTO_CELL_EXPLANATION -->
### Cell 14: training_summary = {
**Explanation:**
- Creates a **training summary dictionary** capturing:
  - File paths (data, model, outputs)
  - Dataset stats (users, items, interactions)
  - Model configuration (embedding dim, layers)
  - Training details (epochs, batch size, steps)
  - Performance:
    - Final loss
    - Loss history
- Saves as a **JSON file** for tracking and reproducibility.


**Possible Output:** Possible output:

Saved: C:\D drive\item_recommendation_model_create\data\ngcf_output\ngcf_training_summary.json


In [14]:
training_summary = {
    "main_data_file": str(MAIN_DATA_FILE),
    "item_catalog_file": str(ITEM_CATALOG_FILE),
    "interaction_file": str(INTERACTION_FILE),
    "model_file": str(MODEL_FILE),
    "top_candidate_file": str(TOP_CANDIDATE_FILE),
    "num_users": int(num_users),
    "num_items": int(num_items),
    "num_interactions": int(len(indexed_interaction_df)),
    "embedding_dim": int(EMBEDDING_DIM),
    "num_layers": int(NUM_LAYERS),
    "epochs": int(EPOCHS),
    "batch_size": int(BATCH_SIZE),
    "steps_per_epoch": int(STEPS_PER_EPOCH),
    "final_loss": float(loss_history[-1]),
    "loss_history": loss_history
}

with open(TRAINING_SUMMARY_FILE, "w", encoding="utf-8") as f:
    json.dump(training_summary, f, ensure_ascii=False, indent=4)

print("Saved:", TRAINING_SUMMARY_FILE)

Saved: C:\D drive\item_recommendation_model_create\data\ngcf_output\ngcf_training_summary.json


<!-- AUTO_CELL_EXPLANATION -->
### Cell 15: check_df = pd.read_csv(TOP_CANDIDATE_FILE)
**Explanation:** This cell executes logic related to `check_df = pd.read_csv(TOP_CANDIDATE_FILE)`.

**Possible Output:** Possible output: text/log/value print, table/HTML render.


In [15]:
check_df = pd.read_csv(TOP_CANDIDATE_FILE)

print("Reloaded shape:", check_df.shape)
print("Unique customers:", check_df["customerId"].nunique())
print("Unique items:", check_df["itemId"].nunique())

display(check_df.head(20))

Reloaded shape: (37000, 6)
Unique customers: 185
Unique items: 229


,customerId,itemId,ngcf_score,ngcf_rank,itemName,category
0,2242,26701,0.999397,1,Nescafe Classic Jar-25gm,Beverage-Hot
1,2242,3554,0.999288,2,Parachute Skin Pure Aloe Vera Gel- 50ml,Personal-Care-Cosmetics
2,2242,7075,0.999280,3,Herman Peanut Butter-340gm,Dairy-Other
3,2242,30788,0.999278,4,Teer Atta 2kg,Pantry-Flour
4,2242,3182,0.999265,5,Starship Chocolate Milk-200ml,Dairy-Milk
5,2242,13922,0.999258,6,Rui Fish (S1)-(MNK),Fish-Fresh
6,2242,4122,0.999236,7,Clear Complete Active Care Shampoo-330ml,Personal-Care-Hair
7,2242,976,0.999234,8,Lata Rice-KG,Pantry-Rice
8,2242,3185,0.999231,9,Marks Milk Shake AS-125,Dairy-Milk
9,2242,125,0.999219,10,Wheel Laundry Soap 125g,Personal-Care-Bath
